# Exploring DuckDB & SQLMesh

Exploring FHIR data infrastructure with DuckDB and SQLMesh

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 02/03/2026 (SGT)   | Martin | Created   | Notebook created. DuckDB exploration | 

# Content

* [DuckDB](#duckdb)

# DuckDB

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import os
from dotenv import load_dotenv

load_dotenv()

True

## Merging files

For systems that cannot handle docker files, use this to create a single merged DB

In [2]:
base_path = "../data/warehouse"
source_files = os.listdir(base_path)
source_files = [f"{base_path}/{file}" for file in source_files]
print(source_files)
target_db_path = "../data/merged_database.duckdb"

con = duckdb.connect(target_db_path)
con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

for i, file_path in enumerate(source_files):
  con.execute(f"ATTACH '{file_path}' AS source{i}")

  tables = con.execute(f"SELECT table_name FROM information_schema.tables WHERE table_catalog = 'source{i}'").fetchall()
  print(tables)
  for table_row in tables:
    table_name = table_row[0]
    table_name = f"bronze.{table_name}"

    con.execute(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM source{i}.{table_name} LIMIT 0")
    con.execute(f"INSERT INTO {table_name} SELECT * FROM source{i}.{table_name}")

for i, file_path in enumerate(source_files):
  con.execute(f"DETACH source{i}")

con.close()

print(f"Data from {source_files} has been merged into {target_db_path}")

['../data/warehouse/mimic_fhir.duckdb', '../data/warehouse/mimic_fhir_condition.duckdb', '../data/warehouse/mimic_fhir_disp_org_obs.duckdb', '../data/warehouse/mimic_fhir_encountered.duckdb', '../data/warehouse/mimic_fhir_location.duckdb', '../data/warehouse/mimic_fhir_obs.duckdb', '../data/warehouse/mimic_fhir_patient.duckdb']
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]
[('fhir_resources',)]
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data from ['../data/warehouse/mimic_fhir.duckdb', '../data/warehouse/mimic_fhir_condition.duckdb', '../data/warehouse/mimic_fhir_disp_org_obs.duckdb', '../data/warehouse/mimic_fhir_encountered.duckdb', '../data/warehouse/mimic_fhir_location.duckdb', '../data/warehouse/mimic_fhir_obs.duckdb', '../data/warehouse/mimic_fhir_patient.duckdb'] has been merged into ../data/merged_database.duckdb


In [ ]:
con = duckdb.connect(source_files[1])
tables = con.sql("SHOW TABLES FROM bronze").df()
print(tables)
con.close()

In [62]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")
dbs = con.sql("SELECT * FROM duckdb_databases").df()
dbs

,database_name,database_oid,path,comment,tags,internal,type,readonly,encrypted,cipher
0,mimic_fhir,651,d:\mads\capstone\mimic-fhir\notebooks\..\data\...,None,{'storage_version': 'v1.0.0+'},False,duckdb,False,False,None


In [63]:
tables = con.sql("SHOW ALL TABLES").df()
tables

,database,schema,name,column_names,column_types,temporary
0,mimic_fhir,bronze,fhir_resources,"[resource_type, resource_id, resource, source_...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR]",False
1,mimic_fhir,bronze,synthea,"[full_url, resource_type, resource, request, s...","[VARCHAR, VARCHAR, JSON, JSON, VARCHAR, VARCHAR]",False
2,mimic_fhir,sqlmesh,_auto_restatements,"[snapshot_name, snapshot_version, next_auto_re...","[VARCHAR, VARCHAR, BIGINT]",False
3,mimic_fhir,sqlmesh,_environment_statements,"[environment_name, plan_id, environment_statem...","[VARCHAR, VARCHAR, VARCHAR]",False
4,mimic_fhir,sqlmesh,_environments,"[name, snapshots, start_at, end_at, plan_id, p...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
5,mimic_fhir,sqlmesh,_intervals,"[id, created_ts, name, identifier, version, st...","[VARCHAR, BIGINT, VARCHAR, VARCHAR, VARCHAR, B...",False
6,mimic_fhir,sqlmesh,_snapshots,"[name, identifier, version, snapshot, kind_nam...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
7,mimic_fhir,sqlmesh,_versions,"[schema_version, sqlglot_version, sqlmesh_vers...","[INTEGER, VARCHAR, VARCHAR]",False


In [64]:
files = con.sql("SELECT DISTINCT source_file FROM bronze.fhir_resources")
files

┌────────────────────────────────┐
│          source_file           │
│            varchar             │
├────────────────────────────────┤
│ Organization.ndjson            │
│ Patient.ndjson                 │
│ EncounterED.ndjson             │
│ ProcedureED.ndjson             │
│ ConditionED.ndjson             │
│ MedicationDispenseED.ndjson    │
│ ObservationVitalSignsED.ndjson │
│ ObservationED.ndjson           │
│ Location.ndjson                │
└────────────────────────────────┘

In [65]:
con.sql("SELECT * FROM bronze.synthea LIMIT 10")

┌───────────────────────────────────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
con.close()

In [ ]:
%load_ext watermark
%watermark